In [23]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_score, recall_score

df_movies = pd.read_csv('movie.csv',nrows=3000)
df_movies['genres'] = df_movies['genres'].fillna('')

df_ratings = pd.read_csv('rating.csv',nrows=3000)

In [24]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_movies['genres'])
similarity_matrix = cosine_similarity(tfidf_matrix)

In [25]:
def get_recommendations(df_movies, similarity_matrix, title, top_n=5):
    if title not in df_movies['title'].values:
        return []
    idx = df_movies[df_movies['title'] == title].index[0]
    sim_scores = list(enumerate(similarity_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    recommendations = [df_movies.iloc[i[0]]['title'] for i in sim_scores]
    return recommendations

In [26]:
def build_ground_truth(df_ratings, df_movies, threshold=4.0):
    user_likes = {}
    for user in df_ratings['userId'].unique():
        liked_movie_ids = df_ratings[(df_ratings['userId'] == user) & (df_ratings['rating'] >= threshold)]['movieId'].tolist()
        liked_titles = df_movies[df_movies['movieId'].isin(liked_movie_ids)]['title'].tolist()
        if liked_titles:
            user_likes[user] = liked_titles
    return user_likes

def evaluate(df_movies, similarity_matrix, ground_truth, top_n=5):
    y_true = []
    y_pred = []

    for user, liked_titles in ground_truth.items():
        for title in liked_titles:
            if title not in df_movies['title'].values:
                continue
            preds = get_recommendations(df_movies, similarity_matrix, title, top_n=top_n)
            trues = set(liked_titles) - {title}

            y_true.extend([1 if item in trues else 0 for item in preds])
            y_pred.extend([1] * len(preds))

    if len(y_true) == 0:
        return 0.0, 0.0

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)

    return precision, recall

In [27]:
ground_truth = build_ground_truth(df_ratings, df_movies)
precision, recall = evaluate(df_movies, similarity_matrix, ground_truth, top_n=5)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

Precision: 0.10
Recall: 1.00


In [35]:
movie_name = "Toy Story " 
recs = get_recommendations(df_movies, similarity_matrix, movie_name, top_n=5)
print(f"Recommendation for film {movie_name}:")
for i, m in enumerate(recs, 1):
    print(f"{i}. {m}")

Recommendation for film Toy Story :
1. Antz 
2. Black Cauldron, The 
3. Lord of the Rings, The 
4. Watership Down 
5. Little Nemo: Adventures in Slumberland 
